In [ ]:
from monai.transforms import Compose, LoadImage
import pandas as pd 
import numpy as np
import os, subprocess, json
import matplotlib.pyplot as plt
from pathlib import PureWindowsPath
import xgboost

In [ ]:

folder_path = "/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTINET"
output_path = "/projects/net_contrast_classification/contrast_phase/total_seg_output"

os.makedirs(output_path, exist_ok=True)

files = []
results = []

# Build correct input paths
for file in data["NiiFile"]:
    path = os.path.join(folder_path, PureWindowsPath(file).name)
    files.append(path)

for f in files:
    basename = os.path.basename(f).replace(".nii.gz", "_phase.json")
    output = os.path.join(output_path, basename)

    # Run TotalSegmentator
    subprocess.run(
        ["totalseg_get_phase", "-i", f, "-o", output, "--fast"],
        check=True
    )

    # Load the JSON we just created
    with open(output) as jf:
        contrast_data = json.load(jf)

    contrast_data["file"] = os.path.basename(f)
    results.append(contrast_data)

df = pd.DataFrame(results)
print(df.head())

df.to_csv("contrast_phase_results.csv", index=False)